# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking / scoring. "Which pages are worth fixing, in order" is a priority score, not a yes/no label — I need pages ordered, not bucketed.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target is **trend_pct**: the observed percent change in clicks between **clicks_prev_30d** and **clicks_last_30d**.  
Not using impression_tier / position_tier: these are just a snapshot and can't tell apart a page that's crashing from one that's steady (same tier, very different urgency).  
Not using trend_direction: it's already a defined category -- someone chose a cutoff to turn the real number into "down/up/stable." trend_pct is the actual observed number underneath it.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

####Success metric: Precision@10
Out of the top 10 pages my model ranks by predicted trend_pct, what fraction actually declined by 20% or worse (trend_pct <= -20%) in the real data?  
K=10 matches what an editor can act on in a week.  
The -20% cutoff matches the real gap in the data between stable pages (~-4% median trend_pct) and declining ones (all below -20%).

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [2]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")  # move from notebooks/ to the repo root

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")

Working dir: /content/flyrank-ml-internship-starter
Starter data found. You're ready.


In [7]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

cols = ["content_id", "trend_pct", "trend_direction", "clicks_last_30d", "clicks_prev_30d",
        "impression_tier", "position_tier"]

print(df[cols].head())
print(f"\nRows: {len(df):,} | one row = one web page (content_id)")

             content_id  trend_pct trend_direction  clicks_last_30d  \
0  content_304f48230142      -41.4            down                2   
1  content_a1fb4e703a9e      -57.7            down                2   
2  content_9aa793d4d895      -60.9            down                1   
3  content_331d6c4de07b      -13.8          stable               22   
4  content_d99b7a2d90ca      -34.7            down               10   

   clicks_prev_30d impression_tier position_tier  
0               13            good      striking  
1                1            good      page_3_5  
2                3            good      page_3_5  
3               17            good        page_1  
4                2            good      page_3_5  

Rows: 30,000 | one row = one web page (content_id)


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule can't do this because it's not one or two signals that
drive decline - it's the combination and interaction of many signals
(search volume, competition, freshness, CTR, position, engagement,
AI traffic share), and which ones matter shifts from page to page.  
That's too tangled to hand-write as an if-statement; a model can learn the combination from data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.